#### **03 — Binning, IV e WoE**

**Objetivo:** ajustar o binning supervisionado no treino, calcular IV/WoE e transformar treino, validação e teste com o mesmo objeto.

**Entrada:** `data/01_prata/*_fe.parquet`  
**Saída:** `data/02_gold/*_woe.parquet`, `binning_process.pkl` e `binning_summary.csv`

O binning é ajustado **somente no treino**.

In [2]:
import pandas as pd
import numpy as np

from modelo_credito.binning import (
    binning_summary,
    build_transformed_dataset,
    fit_binning_process,
    save_binning_process,
)

pd.set_option("display.max_columns", None)


##### **1. Carga das bases com features**

In [3]:
treino = pd.read_parquet("../data/01_prata/treino_fe.parquet")
validacao = pd.read_parquet("../data/01_prata/validacao_fe.parquet")
teste = pd.read_parquet("../data/01_prata/teste_fe.parquet")

print("treino:", treino.shape)
print("validacao:", validacao.shape)
print("teste:", teste.shape)

treino: (18000, 38)
validacao: (6000, 38)
teste: (6000, 38)


#### **2. Definição das variáveis**

`ID` e `SAFRA` são metadados. `default` é o target. As demais colunas entram no binning.

In [4]:
metadata = ["ID", "SAFRA"]
target = "default"

categorical_variables = [
    "SEX", "EDUCATION", "MARRIAGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
]

variable_names = [
    col for col in treino.columns
    if col not in metadata + [target]
]

print("Quantidade de variáveis:", len(variable_names))
print(variable_names)

Quantidade de variáveis: 35
['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'SUM_BILL', 'AVG_BILL', 'MAX_BILL', 'SUM_PAY', 'AVG_PAY', 'MAX_PAY', 'UTILIZACAO', 'UTIL_MEDIA', 'PCT_PAGAMENTO', 'FREQ_ATRASO', 'MAX_ATRASO', 'LIMIT_PER_CAPITA']


#### **3. Ajuste do binning no treino**

In [5]:
binning_process = fit_binning_process(
    treino,
    variable_names,
    categorical_variables=categorical_variables,
    target=target,
    min_prebin_size=0.05,
    max_n_bins=5,
)


,variable_names,"['LIMIT_BAL', 'SEX', ...]"
,max_n_bins,5
,categorical_variables,"['SEX', 'EDUCATION', ...]"
,max_n_prebins,20
,min_prebin_size,0.05
,min_n_bins,None
,min_bin_size,None
,max_bin_size,None
,max_pvalue,None
,max_pvalue_policy,'consecutive'
,selection_criteria,None


#### **4. Resumo de IV e qualidade do binning**

In [6]:
binning_summary_df = binning_summary(binning_process)
binning_summary_df.to_csv("../data/02_gold/binning_summary.csv", index=False)

binning_summary_df.head(15)


,name,dtype,status,selected,n_bins,iv,js,gini,quality_score
5,PAY_0,categorical,OPTIMAL,True,5,0.833657,0.094231,0.428338,0.071618
32,FREQ_ATRASO,numerical,OPTIMAL,True,5,0.810085,0.094472,0.444401,0.282526
33,MAX_ATRASO,numerical,OPTIMAL,True,4,0.709273,0.085492,0.428707,0.515249
6,PAY_2,categorical,OPTIMAL,True,4,0.503322,0.059104,0.302253,0.604219
7,PAY_3,categorical,OPTIMAL,True,4,0.376393,0.044689,0.259449,0.793162
8,PAY_4,categorical,OPTIMAL,True,4,0.343313,0.040244,0.229274,0.751601
9,PAY_5,categorical,OPTIMAL,True,4,0.334614,0.03909,0.226833,0.784267
10,PAY_6,categorical,OPTIMAL,True,4,0.263786,0.031338,0.205864,0.702488
17,PAY_AMT1,numerical,OPTIMAL,True,5,0.175904,0.021601,0.219936,0.560901
26,SUM_PAY,numerical,OPTIMAL,True,5,0.171307,0.021066,0.22469,0.634785


#### **5. Transformação WoE**

O mesmo `binning_process` é aplicado em treino, validação e teste.

In [7]:
treino_woe = build_transformed_dataset(
    binning_process, treino, variable_names, metadata, target, metric="woe"
)
validacao_woe = build_transformed_dataset(
    binning_process, validacao, variable_names, metadata, target, metric="woe"
)
teste_woe = build_transformed_dataset(
    binning_process, teste, variable_names, metadata, target, metric="woe"
)

print("treino_woe:", treino_woe.shape, "| nulos:", int(treino_woe.isna().sum().sum()))
print("validacao_woe:", validacao_woe.shape, "| nulos:", int(validacao_woe.isna().sum().sum()))
print("teste_woe:", teste_woe.shape, "| nulos:", int(teste_woe.isna().sum().sum()))


treino_woe: (18000, 38) | nulos: 0
validacao_woe: (6000, 38) | nulos: 0
teste_woe: (6000, 38) | nulos: 0


#### **6. Gravação dos objetos e bases WoE**

In [8]:
treino_woe.to_parquet("../data/02_gold/treino_woe.parquet")
validacao_woe.to_parquet("../data/02_gold/validacao_woe.parquet")
teste_woe.to_parquet("../data/02_gold/teste_woe.parquet")

save_binning_process(binning_process, "../data/02_gold/binning_process.pkl")

print("Arquivos gravados em data/02_gold")


Arquivos gravados em data/02_gold


#### **7. Conferência de uma variável**

In [9]:
optb = binning_process.get_binned_variable("PAY_0")
optb.binning_table.build()

,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,[-2.0],1371,0.076167,1195,176,0.128373,0.710241,0.031160,0.003815
1,[0.0],8741,0.485611,7595,1146,0.131106,0.686036,0.186760,0.022898
2,[-1.0],3556,0.197556,2897,659,0.185321,0.275531,0.013880,0.001729
3,[1.0],2347,0.130389,1532,815,0.347252,-0.574036,0.049236,0.006071
4,[2.0],1985,0.110278,631,1354,0.682116,-1.968689,0.552622,0.059717
5,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
6,Missing,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
Totals,,18000,1.000000,13850,4150,0.230556,,0.833657,0.094231


In [10]:
treino_woe.head()

,ID,SAFRA,default,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,SUM_BILL,AVG_BILL,MAX_BILL,SUM_PAY,AVG_PAY,MAX_PAY,UTILIZACAO,UTIL_MEDIA,PCT_PAGAMENTO,FREQ_ATRASO,MAX_ATRASO,LIMIT_PER_CAPITA
0,1,200510,1,-0.637758,0.058133,-0.064066,-0.082203,-0.289752,-1.968689,-1.410564,0.399019,0.355912,0.098482,0.064377,-0.022995,-0.035308,-0.003742,-0.061010,-0.104462,-0.054363,-0.636534,-0.100966,-0.493180,-0.470681,-0.364949,-0.351205,-0.053502,-0.053502,-0.048256,-0.448876,-0.448876,-0.252400,0.415341,0.497466,-0.671710,-0.699341,-1.041892,-0.366799
1,2,200510,1,0.017844,0.058133,-0.064066,0.073595,0.104268,0.275531,-1.410564,0.305270,0.246864,0.226628,-1.232284,-0.022995,-0.081630,-0.003742,0.406697,0.337217,0.409344,-0.636534,-0.100966,0.015049,-0.043929,-0.364949,0.106082,0.317342,0.317342,-0.048256,-0.448876,-0.448876,-0.252400,0.415341,0.497466,-0.671710,-0.699341,-1.041892,0.157529
2,3,200510,0,-0.161543,0.058133,-0.064066,0.073595,0.181117,0.686036,0.420354,0.305270,0.246864,0.226628,0.221809,-0.111511,-0.035308,-0.003742,0.030980,-0.032032,-0.044797,-0.017196,-0.100966,0.015049,-0.043929,-0.085945,0.369648,0.013956,0.013956,-0.048256,-0.155030,-0.155030,0.162548,0.117988,0.497466,0.110056,0.763302,0.856228,-0.124367
3,4,200510,0,-0.161543,0.058133,-0.064066,-0.082203,0.029218,0.686036,0.420354,0.305270,0.246864,0.226628,0.221809,0.061451,-0.035308,0.092901,-0.025638,-0.032032,-0.044797,-0.017196,0.091827,0.015049,-0.043929,-0.085945,-0.103595,0.013956,0.013956,0.034861,-0.155030,-0.155030,-0.252400,-0.202729,-0.168092,0.110056,0.763302,0.856228,-0.366799
4,5,200510,0,-0.161543,-0.079805,-0.064066,-0.082203,-0.139186,0.275531,0.420354,0.399019,0.246864,0.226628,0.221809,-0.022995,-0.035308,-0.094716,-0.025638,-0.032032,-0.044797,-0.017196,0.766122,0.582085,0.561955,-0.085945,-0.103595,0.013956,0.013956,0.034861,0.466820,0.466820,0.681599,0.415341,0.192644,0.110056,0.763302,0.856228,-0.366799
